# GLCMaeV8 Inference Pipeline

Load a trained `GLCMaeV8` checkpoint, run inference on welding robot data, and produce anomaly verdicts with phase-aware scoring.

**Data split:**
- **Train set** — used to build robot/program codebooks only
- **Val set** (normal only) — used for MAD threshold calibration
- **Test set** (50% normal / 50% anomaly, 10 types) — used for evaluation

**Pipeline:**
1. Load trained model checkpoint
2. Load train CSV → build codebooks
3. Load val CSV → preprocess into file-windows
4. Load test CSV + test_labels → preprocess into file-windows
5. Score val + test sets with `Reconf8LocalPreResult`
6. Fit threshold from val normal scores via `Reconf8LocalThresholdCreator`
7. Apply threshold to test set → evaluate per-type + overall F1 / ROC-AUC
8. Per-sensor heatmap analysis
9. Visualization
10. Summary report

In [ ]:
# %% Imports
import sys
import csv
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add project root to path
project_root = Path.cwd().parent.parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.anomaly_detection.models import (
    GLCMaeV8,
    ReconstructionModels,
)
from app.anomaly_detection.inference.recon_v8.local import (
    Reconf8LocalPreResult,
    Reconf8LocalThresholdCreator,
    Reconf8LocalApplier,
)

print("All imports OK")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

## 1. Load Trained Model Checkpoint

Load the model state_dict + config saved by the training notebook.

In [ ]:
# Path to trained checkpoint from Training-GLCReconV8 notebook
CKPT_DIR   = project_root / "artifacts" / "mae_v8_checkpoint"
MODEL_SLUG = "GLCMaeV8"
MODEL_FILE = f"model_{MODEL_SLUG}_best.pt"
META_FILE  = f"metadata_{MODEL_SLUG}_best.json"
EXTRA_FILE = CKPT_DIR / "training_extra.json"

if not (CKPT_DIR / MODEL_FILE).exists():
    print(f"Checkpoint not found at {CKPT_DIR / MODEL_FILE}")
    print("Run the Training-GLCReconV8 notebook first to train and save a model.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = GLCMaeV8.load_from_checkpoint(
        checkpoint=str(CKPT_DIR),
        model_file=MODEL_FILE,
        metadata_file=META_FILE,
        device=str(device),
    )
    print(f"Model restored to {device}: {model.name}")
    print(f"  Params:            {sum(p.numel() for p in model.parameters()):,}")
    print(f"  num_sensors:       {model.num_sensors}")
    print(f"  seq_len:           {model.seq_len}")
    print(f"  num_robots:        {model.num_robots}")
    print(f"  num_programs:      {model.num_programs}")
    print(f"  mask_ratio:        {model.mask_ratio}")
    print(f"  use_context_emb:   {model.use_context_embeddings}")

    # Load training extras (history, codebooks, threshold)
    extras = {}
    if EXTRA_FILE.exists():
        with open(EXTRA_FILE) as f:
            extras = json.load(f)
        print(f"  Training extras loaded: history ({len(extras.get('history', {}).get('loss', []))} epochs), "
              f"robot_codebook ({len(extras.get('robot_codebook', {}))} codes), "
              f"program_codebook ({len(extras.get('program_codebook', {}))} codes)")
    robot_codebook   = extras.get("robot_codebook", {})
    program_codebook = extras.get("program_codebook", {})
    print(f"  robot_codebook:    {robot_codebook}")
    print(f"  program_codebook:  {dict(list(program_codebook.items())[:5])} ...")

## 2. Load Train + Val + Test CSV Data

Three separate CSV files (generated by Sprint 16).
Group by `cycle_id` into macro-windows (seq_len=240).
Build robot/program codebooks from **training extras** saved at training time.

In [ ]:
import pandas as pd

DATA_DIR        = project_root / "data" / "generated"
TRAIN_CSV       = DATA_DIR / "train_welding_robot.csv"
VAL_CSV         = DATA_DIR / "val_welding_robot.csv"
TEST_CSV        = DATA_DIR / "test_welding_robot.csv"
TEST_LABELS_CSV = DATA_DIR / "test_labels.csv"

SENSOR_COLS = ["wire_feed_speed_mm_min", "wire_temperature_c", "welding_current_a"]
SEQ_LEN     = 240

def load_csv_to_file_records(csv_path):
    """Load CSV -> group by cycle_id -> build file_records list."""
    print(f"Reading {csv_path}...")
    df = pd.read_csv(csv_path)
    print(f"  Rows: {len(df):,}, cycle_ids: {df['cycle_id'].nunique()}")

    records = []
    for cid, group in df.groupby("cycle_id"):
        group = group.sort_values("serial_number")
        file_arr = np.array([group[col].values for col in SENSOR_COLS], dtype=np.float32)  # [C, T]
        T = file_arr.shape[1]
        if T >= SEQ_LEN:
            window = file_arr[:, :SEQ_LEN]
        else:
            pad = np.zeros((len(SENSOR_COLS), SEQ_LEN - T), dtype=np.float32)
            window = np.hstack([file_arr, pad])
        records.append({
            "cycle_id":      cid,
            "window":        window,
            "robot_code":    str(group["robot_code"].iloc[0]),
            "program_number": int(group["program_number"].iloc[0]),
            "anomaly_type":  str(group["anomaly_type"].iloc[0]),
        })
    return records

print("=" * 50)
print("LOADING DATA")
print("=" * 50)

train_records = load_csv_to_file_records(TRAIN_CSV)
val_records   = load_csv_to_file_records(VAL_CSV)
test_records  = load_csv_to_file_records(TEST_CSV)

# Use codebooks saved at training time (ensures same index mapping)
if not robot_codebook:
    print("WARNING: robot_codebook empty — rebuilding from train set")
    all_robot_codes  = sorted(set(r["robot_code"] for r in train_records))
    all_program_nums = sorted(set(str(r["program_number"]) for r in train_records))
    robot_codebook   = {code: i for i, code in enumerate(all_robot_codes)}
    program_codebook = {pn:   i for i, pn   in enumerate(all_program_nums)}
# Extend for any unseen codes in val/test
for r in val_records + test_records:
    rc, pn = str(r["robot_code"]), str(r["program_number"])
    if rc not in robot_codebook:
        robot_codebook[rc] = len(robot_codebook)
    if pn not in program_codebook:
        program_codebook[pn] = len(program_codebook)

# Load test labels
test_labels = pd.read_csv(TEST_LABELS_CSV)
label_map   = dict(zip(test_labels["cycle_id"].astype(str), test_labels["anomaly_type"]))

n_val_normal   = sum(1 for r in val_records)
n_test_normal  = sum(1 for r in test_records if r["anomaly_type"] == "normal")
n_test_anomaly = sum(1 for r in test_records if r["anomaly_type"] != "normal")
print(f"\nTrain: {len(train_records)} files, {len(robot_codebook)} robots, {len(program_codebook)} programs")
print(f"Val (normal):   {n_val_normal} files")
print(f"Test (normal):  {n_test_normal} files")
print(f"Test (anomaly): {n_test_anomaly} files")
print(f"Example: {test_records[0]['cycle_id']}, robot={test_records[0]['robot_code']}, "
      f"prog={test_records[0]['program_number']}, anomaly={test_records[0]['anomaly_type']}")

## 3. Build Macro-Window Records

Each file record is converted to a list of overlapping 240-step windows with 16-step stride.
The resulting dict mirrors the format expected by `Reconf8LocalPreResult.score_batch()`.

In [ ]:
def build_pre_result_records(records_list, seq_len=SEQ_LEN, patch_stride=16):
    """Convert file_records -> pre_result_records for score_batch()."""
    result = []
    for rec in records_list:
        window = rec["window"]   # [C, T]
        _, T   = window.shape
        windows_list = []
        serials_list = []
        for w in range(0, max(T - seq_len + 1, 1), patch_stride):
            if w + seq_len <= T:
                windows_list.append(window[:, w : w + seq_len])
                serials_list.append(w)
        if not windows_list:
            windows_list.append(window[:, :seq_len])
            serials_list.append(0)
        result.append({
            "robot_code":     rec["robot_code"],
            "program_number": rec["program_number"],
            "start_time":     str(rec["cycle_id"]),
            "windows":        np.stack(windows_list),
            "serials":        np.array(serials_list, dtype=np.int64),
        })
    return result

val_pre_records  = build_pre_result_records(val_records)
test_pre_records = build_pre_result_records(test_records)

print(f"Val pre-result records:  {len(val_pre_records)}")
print(f"Test pre-result records: {len(test_pre_records)}")
print(f"Val record[0]:  windows={val_pre_records[0]['windows'].shape}, "
      f"file: {len(val_pre_records[0]['windows'])} windows")
print(f"Test record[0]: windows={test_pre_records[0]['windows'].shape}")

## 4. Score Val + Test Sets

**Flow:**
1. `Reconf8LocalPreResult.score_batch()` with codebooks → per-file p95 scores (global + rampup + active)
2. `Reconf8LocalThresholdCreator` calibrated **only on val set** (100% normal)
3. Apply threshold to test set → OR-logic verdict

In [ ]:
metadata   = model.get_metadata()
state_dict = model.state_dict()
device_str = str(device)

pre_result_engine = Reconf8LocalPreResult(
    model_version=ReconstructionModels.GLC_MAE_V8,
    metadata=metadata,
    state_dict=state_dict,
    quantile=0.95,
    k_skip=0,
    device=device_str,
    robot_codebook=robot_codebook,
    program_codebook=program_codebook,
)

print("Scoring validation set (normal only)...")
val_pre_scores = pre_result_engine.score_batch(val_pre_records)

print("Scoring test set (normal + anomaly)...")
test_pre_scores = pre_result_engine.score_batch(test_pre_records)

print(f"\nVal scored:  {len(val_pre_scores)} files")
print(f"Test scored: {len(test_pre_scores)} files")
print(f"Keys in a result: {list(val_pre_scores[0].keys())[:8]}")

In [ ]:
# ── Fit multi-score MAD threshold from VAL set only ──
num_sensors    = model.num_sensors
pq_sensor_keys = Reconf8LocalThresholdCreator.make_per_sensor_keys(0.95, num_sensors)
SCORE_KEYS     = ["p95_score", "p95_score_rampup", "p95_score_active"] + pq_sensor_keys

tc = Reconf8LocalThresholdCreator(
    quantile=0.95,
    score_keys=SCORE_KEYS,
)
tc.fit_from_pre_results(val_pre_scores)

print(f"Multi-score thresholds (from {len(val_pre_scores)} val files):")
for key, thresh in tc.thresholds.items():
    print(f"  {key}: {thresh:.6f}")

# ── Apply to test set with OR-logic ──
applier = Reconf8LocalApplier({
    "global_threshold":  1e10,
    "score_thresholds":  tc.thresholds,
}, quantile=0.95)
test_applied = applier.apply(test_pre_scores)

print(f"Test verdict distribution:")
n_pred_anom = sum(1 for r in test_applied if r["is_file_anomaly"])
n_pred_norm = len(test_applied) - n_pred_anom
print(f"  Predicted normal:  {n_pred_norm}")
print(f"  Predicted anomaly: {n_pred_anom}")

from collections import Counter
trigger_counts = Counter()
for r in test_applied:
    for t in r.get("triggered_by", []):
        trigger_counts[t] += 1
print(f"Trigger distribution:")
for key, count in trigger_counts.most_common():
    print(f"  {key}: {count}")

## 5. Val Score Statistics

Sanity-check the calibration set score distribution.

In [ ]:
val_p95 = np.array([r["p95_score"] for r in val_pre_scores], dtype=float)

stats = {
    "count":  int(val_p95.size),
    "mean":   float(val_p95.mean()),
    "std":    float(val_p95.std(ddof=1)) if val_p95.size > 1 else 0.0,
    "min":    float(val_p95.min()),
    "p5":     float(np.percentile(val_p95, 5)),
    "p25":    float(np.percentile(val_p95, 25)),
    "median": float(np.percentile(val_p95, 50)),
    "p75":    float(np.percentile(val_p95, 75)),
    "p95":    float(np.percentile(val_p95, 95)),
    "max":    float(val_p95.max()),
}

print("Val set p95_score statistics:")
for key in ["count", "mean", "std", "min", "p5", "p25", "median", "p75", "p95", "max"]:
    print(f"{key:>8}: {stats[key]:.6f}" if key != "count" else f"{key:>8}: {stats[key]}")

## 6. Evaluate: Per-Anomaly-Type + Overall Metrics

Precision, Recall, F1, and ROC-AUC for each of the 10 anomaly types.

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

# ── Build ground-truth + prediction arrays ──
y_true_anomaly = np.array([
    1 if r["anomaly_type"] != "normal" else 0
    for r in test_records
])
y_pred = np.array([
    1 if r["is_file_anomaly"] else 0
    for r in test_applied
])
y_score = np.array([r["p95_score"] for r in test_applied])
anomaly_types_list = [r["anomaly_type"] for r in test_records]

# ── Per-type metrics ──
unique_types = sorted(set(anomaly_types_list))
print("=" * 70)
print("PER-ANOMALY-TYPE EVALUATION")
print("=" * 70)
print(f"{'Type':<22} {'Count':>6} {'Prec':>8} {'Recall':>8} {'F1':>8} {'ROC-AUC':>8}")
print("-" * 70)

per_type_results = {}
for at in unique_types:
    y_true_bin = np.array([1 if t == at else 0 for t in anomaly_types_list])
    if y_true_bin.sum() == 0:
        continue
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred, average="binary", zero_division=0
    )
    try:
        roc = roc_auc_score(y_true_bin, y_score)
    except ValueError:
        roc = float("nan")
    per_type_results[at] = {"precision": prec, "recall": rec, "f1": f1, "roc_auc": roc}
    print(f"{at:<22} {y_true_bin.sum():>6} {prec:>8.4f} {rec:>8.4f} {f1:>8.4f} {roc:>8.4f}")

# ── Overall (anomaly vs normal) ──
print("\n" + "=" * 70)
print("OVERALL (Anomaly vs Normal)")
print("=" * 70)
overall_prec, overall_rec, overall_f1, _ = precision_recall_fscore_support(
    y_true_anomaly, y_pred, average="binary", zero_division=0
)
try:
    overall_roc = roc_auc_score(y_true_anomaly, y_score)
except ValueError:
    overall_roc = float("nan")

tp = int(((y_true_anomaly == 1) & (y_pred == 1)).sum())
fp = int(((y_true_anomaly == 0) & (y_pred == 1)).sum())
fn = int(((y_true_anomaly == 1) & (y_pred == 0)).sum())
tn = int(((y_true_anomaly == 0) & (y_pred == 0)).sum())
accuracy = (tp + tn) / len(y_true_anomaly)

print(f"  Accuracy:  {accuracy:.4f}  ({tp + tn}/{len(y_true_anomaly)})")
print(f"  Precision: {overall_prec:.4f}")
print(f"  Recall:    {overall_rec:.4f}")
print(f"  F1 Score:  {overall_f1:.4f}")
print(f"  ROC-AUC:   {overall_roc:.4f}")
print()
print("Confusion Matrix:")
print(f"              Pred Normal   Pred Anomaly")
print(f"  True Normal   {tn:>4d}           {fp:>4d}")
print(f"  True Anomaly  {fn:>4d}           {tp:>4d}")

In [ ]:
# ── Target comparison ──
targets = {
    "spike": 0.98, "arc_blow": 0.98, "gas_blockage": 0.95,
    "thermal_drift": 0.90, "wire_stutter": 0.90, "sticky_tc": 0.85,
    "flatline": 0.85, "phase_shift": 0.80,
    "cable_wear": 0.85, "overshoot_degradation": 0.75,
}
print("=" * 70)
print("TARGET COMPARISON")
print("=" * 70)
for at, target in targets.items():
    if at in per_type_results:
        actual = per_type_results[at]["recall"]
        status = "OK" if actual >= target else "MISS"
        print(f"  {at:<22} target={target:.2f} actual={actual:.4f} {status}")

## 7. Visualize Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

val_scores  = np.array([r["p95_score"] for r in val_pre_scores])
test_scores = np.array([r["p95_score"] for r in test_applied])
test_is_anom = np.array([
    1 if r["anomaly_type"] != "normal" else 0
    for r in test_records
])

# Histogram: val (normal) vs test normal vs test anomaly
ax = axes[0]
ax.hist(val_scores, bins=30, alpha=0.7,
        label=f"Val normal (n={len(val_scores)})", color="steelblue")
ax.hist(test_scores[test_is_anom == 0], bins=20, alpha=0.5,
        label=f"Test normal (n={(test_is_anom == 0).sum()})", color="skyblue")
ax.hist(test_scores[test_is_anom == 1], bins=10, alpha=0.7,
        label=f"Test anomaly (n={(test_is_anom == 1).sum()})", color="crimson")
for key, thresh in tc.thresholds.items():
    ax.axvline(x=thresh, ls="--", alpha=0.5, label=f"{key}={thresh:.4f}")
ax.set_xlabel("Anomaly Score (p95)")
ax.set_ylabel("Files")
ax.set_title("Score Distribution")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Box plot: normal vs anomaly
ax = axes[1]
bp = ax.boxplot(
    [test_scores[test_is_anom == 0], test_scores[test_is_anom == 1]],
    labels=["Normal", "Anomaly"], patch_artist=True,
)
bp["boxes"][0].set_facecolor("steelblue")
bp["boxes"][1].set_facecolor("crimson")
ax.set_ylabel("Anomaly Score")
ax.set_title("Test Set: Normal vs Anomaly")
ax.grid(True, alpha=0.3)

# Ranked scores
ax = axes[2]
sort_idx     = np.argsort(test_scores)[::-1]
sorted_scores = test_scores[sort_idx]
sorted_labels = test_is_anom[sort_idx]
ax.plot(sorted_scores, marker=".", markersize=3, color="steelblue")
for key, thresh in tc.thresholds.items():
    ax.axhline(y=thresh, linestyle="--", alpha=0.5, label=f"{key}={thresh:.4f}")
ax.fill_between(range(len(sorted_scores)), sorted_scores, min(sorted_scores),
                where=(sorted_labels == 1),
                color="crimson", alpha=0.3, label="Ground-truth anomaly")
ax.set_xlabel("File rank (highest score first)")
ax.set_ylabel("Anomaly Score")
ax.set_title("Ranked Test Scores")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Per-Sensor Heatmap Analysis

Compare timestep-level reconstruction error heatmaps between normal and anomaly files.

In [ ]:
# Pick one normal and one anomaly test file for heatmap comparison
normal_idx  = next(i for i, r in enumerate(test_records) if r["anomaly_type"] == "normal")
anomaly_idx = next(i for i, r in enumerate(test_records) if r["anomaly_type"] != "normal")

normal_result  = test_applied[normal_idx]
anomaly_result = test_applied[anomaly_idx]

SENSOR_NAMES = ["wire_feed_speed_mm_min", "wire_temperature_c", "welding_current_a"]
sensors_in_results = [k[len("heatmap_"):] for k in normal_result if k.startswith("heatmap_")]

fig, axes = plt.subplots(len(sensors_in_results), 2, figsize=(16, 3 * len(sensors_in_results)))
if len(sensors_in_results) == 1:
    axes = axes[None, :]

for c, s in enumerate(sensors_in_results):
    normal_heatmap  = np.array(normal_result.get(f"heatmap_{s}", []))
    anomaly_heatmap = np.array(anomaly_result.get(f"heatmap_{s}", []))

    t_normal  = np.arange(len(normal_heatmap))
    t_anomaly = np.arange(len(anomaly_heatmap))

    axes[c, 0].plot(t_normal,  normal_heatmap,  color="steelblue", lw=1.5)
    axes[c, 0].set_title(f"{s} — Normal file")
    axes[c, 0].set_ylabel("Recon error")
    axes[c, 0].grid(True, alpha=0.3)

    axes[c, 1].plot(t_anomaly, anomaly_heatmap, color="crimson", lw=1.5)
    axes[c, 1].set_title(f"{s} — Anomaly file ({anomaly_result['anomaly_type']})"
                          if hasattr(anomaly_result, 'get') and 'anomaly_type' in anomaly_result
                          else f"{s} — Anomaly file")
    axes[c, 1].set_ylabel("Recon error")
    axes[c, 1].grid(True, alpha=0.3)

plt.suptitle(
    f"Normal ({normal_result['robot_code']}/{normal_result['program_number']}) vs "
    f"Anomaly ({anomaly_result['robot_code']}/{anomaly_result['program_number']})",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(f"Normal  file: p95={normal_result['p95_score']:.4f}  is_anomaly={normal_result['is_file_anomaly']}")
anomaly_score_key = "p95_score"
print(f"Anomaly file: p95={anomaly_result[anomaly_score_key]:.4f}  is_anomaly={anomaly_result['is_file_anomaly']}")
print(f"  triggered_by: {anomaly_result.get('triggered_by', [])}")

## 9. Phase-Score Distribution

Visualize the distribution of phase-specific scores (rampup / active) for the test set.

In [ ]:
if "p95_score_rampup" in test_applied[0]:
    rampup_scores = np.array([r.get("p95_score_rampup", 0.0) for r in test_applied])
    active_scores = np.array([r.get("p95_score_active", 0.0) for r in test_applied])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, scores, title in [
        (axes[0], rampup_scores, "Rampup Phase Score"),
        (axes[1], active_scores, "Active Phase Score"),
    ]:
        ax.hist(scores[test_is_anom == 0], bins=20, alpha=0.7,
                color="steelblue", label="Normal")
        ax.hist(scores[test_is_anom == 1], bins=10, alpha=0.7,
                color="crimson", label="Anomaly")
        key = "p95_score_rampup" if "Rampup" in title else "p95_score_active"
        if key in tc.thresholds:
            ax.axvline(x=tc.thresholds[key], color="green", ls="--", lw=2,
                       label=f"threshold={tc.thresholds[key]:.4f}")
        ax.set_xlabel("Score")
        ax.set_ylabel("Files")
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Phase scores not available in pre-results (check score_batch output keys)")

## 10. Summary Report

In [ ]:
print("=" * 70)
print("GLCMaeV8 INFERENCE SUMMARY")
print("=" * 70)
print(f"  Model:              {model.name}")
print(f"  Checkpoint:         {CKPT_DIR / MODEL_FILE}")
print(f"  Device:             {device}")
print()
print("--- DATA SPLIT ---")
print(f"  Train files:        {len(train_records)}")
print(f"  Val files (normal): {len(val_records)}")
print(f"  Test files:         {len(test_records)}")
print(f"    Normal:  {n_test_normal}")
print(f"    Anomaly: {n_test_anomaly} ({len(unique_types) - 1} types)")
print()
print("--- THRESHOLDS (from val set) ---")
for key, thresh in tc.thresholds.items():
    med   = tc._medians.get(key, float('nan'))
    mad_v = tc._mads.get(key, float('nan'))
    print(f"  {key}: {thresh:.6f}  (median={med:.4f}, MAD={mad_v:.4f})")
print()
print("--- OVERALL METRICS (test set) ---")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {overall_prec:.4f}")
print(f"  Recall:    {overall_rec:.4f}")
print(f"  F1 Score:  {overall_f1:.4f}")
print(f"  ROC-AUC:   {overall_roc:.4f}")
print()
print("--- PER-TYPE RECALL ---")
for at in sorted(per_type_results.keys()):
    r = per_type_results[at]
    target = targets.get(at, None)
    status = "OK" if target and r["recall"] >= target else "MISS" if target else ""
    target_str = f" (target {target:.0%})" if target else ""
    print(f"  {at:<22} recall={r['recall']:.2%}  f1={r['f1']:.2%}  roc={r['roc_auc']:.2%}{target_str} {status}")
print()
print("--- CONFUSION MATRIX ---")
print(f"              Pred Normal   Pred Anomaly")
print(f"  True Normal   {tn:>4d}           {fp:>4d}")
print(f"  True Anomaly  {fn:>4d}           {tp:>4d}")
print("=" * 70)